## 🔧 Path Setup & Project Root

In [2]:
import sys
import os
from pathlib import Path

# Resolve project root (one level up from notebooks/)
PROJECT_ROOT = Path("../").resolve()

# Add project root to Python path if not present
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f"✅ Project root added to sys.path: {PROJECT_ROOT}")
print(f"📁 Project root  : {PROJECT_ROOT}")
print(f"📁 Working dir   : {Path.cwd()}")
print(f"🐍 Python version: {sys.version.split()[0]}")

✅ Project root added to sys.path: C:\santosh\MediQwen
📁 Project root  : C:\santosh\MediQwen
📁 Working dir   : c:\santosh\MediQwen\notebooks
🐍 Python version: 3.14.2


## 🧪 Component-Wise Verification of `nodes.py`

In this section, we test each node function in isolation using mock `AgentState` payloads. 
This allows us to verify guardrail integration, safety refusals, risk overrides, and Qwen generation outputs independently before passing state through the full LangGraph orchestrator.

In [2]:
from agent.state import AgentState
from agent.nodes import (
    run_dialogue_manager,
    run_risk_classification,
    run_multimodal_processing,
    run_vector_search,
    run_web_search_tool,
    safe_refusal_node,
    run_qwen_generation
)

# Helper to build a clean baseline state
def create_test_state(**kwargs) -> AgentState:
    base: AgentState = {
        "user_query": "",
        "image_path": None,
        "process_image": False,
        "processed_image_payload": None,
        "is_online": False,
        "dialog_state": "chat",
        "requires_retrieval": False,
        "clinical_subject": None,
        "followup_pending": False,
        "risk_level": "LOW",
        "risk_metadata": {},
        "is_safe": True,
        "intent": "general",
        "top_retrieval_distance": 0.0,
        "retrieved_context": [],
        "context_sources": [],
        "retrieval_query": None,
        "agent_response": "",
        "routing_trace": [],
        "show_risk_badge": False
    }
    base.update(kwargs)
    return base

print("--- [TEST 1A: Multimodal Vague Query] ---")
HIVES_IMAGE_PATH = PROJECT_ROOT / "data" / "test_assets" / "hives.jpg"
state_img = create_test_state(user_query="What could this be?", image_path=HIVES_IMAGE_PATH)
res_img = run_dialogue_manager(state_img)
print(f"Is Safe: {res_img.get('is_safe')} | Refusal: {res_img.get('agent_response')}")
print(f"State: {res_img.get('dialog_state')} | Process Image: {res_img.get('process_image')} | Retrieval: {res_img.get('requires_retrieval')}")

assert res_img['is_safe'] == True
assert res_img['dialog_state'] == "multimodal_triage"
assert res_img['process_image'] == True

print("\n--- [TEST 1B: Explicit Clinical Retrieval] ---")
state_rag = create_test_state(user_query="What are the symptoms and treatment for angina?")
res_rag = run_dialogue_manager(state_rag)
print(f"State: {res_rag['dialog_state']} | Retrieval: {res_rag['requires_retrieval']} | Subject: {res_rag['clinical_subject']}")
assert res_rag['requires_retrieval'] == True

print("\n--- [TEST 1C: Safety Intercept (Code Request)] ---")
state_code = create_test_state(user_query="Write a python script to analyze patient records")
res_code = run_dialogue_manager(state_code)
print(f"Is Safe: {res_code['is_safe']} | State: {res_code['dialog_state']}")
assert res_code['is_safe'] == False
assert res_code['dialog_state'] == "blocked"

c:\santosh\MediQwen\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026-09-12 11:51:00,001 - INFO - ⚡ MediQwen Guardrails Safety Engine Online.
2026-09-12 11:51:00,002 - INFO - ⚡ Accumulative Rules Engine Active. Safety Enforcement: True
2026-09-12 11:51:00,003 - INFO - ⚡ MediQwen Guardrails Safety Engine Online.
2026-09-12 11:51:00,004 - INFO - Initializing Persistent ChromaDB client at: C:\santosh\MediQwen\chroma_db


🔍 Initializing embedding model: BAAI/bge-large-en-v1.5
🔧 Using device: auto


2026-09-12 11:51:30,547 - INFO - HTTP Request: HEAD https://huggingface.co/BAAI/bge-large-en-v1.5/resolve/main/modules.json "HTTP/1.1 307 Temporary Redirect"
2026-09-12 11:51:30,856 - INFO - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/BAAI/bge-large-en-v1.5/d4aa6901d3a41ba39fb536a557fa166f842b0e09/modules.json "HTTP/1.1 200 OK"
2026-09-12 11:51:31,168 - INFO - HTTP Request: HEAD https://huggingface.co/BAAI/bge-large-en-v1.5/resolve/main/config_sentence_transformers.json "HTTP/1.1 307 Temporary Redirect"
2026-09-12 11:51:31,465 - INFO - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/BAAI/bge-large-en-v1.5/d4aa6901d3a41ba39fb536a557fa166f842b0e09/config_sentence_transformers.json "HTTP/1.1 200 OK"
2026-09-12 11:51:31,482 - INFO - Loading SentenceTransformer model from BAAI/bge-large-en-v1.5.
2026-09-12 11:51:31,689 - INFO - HTTP Request: HEAD https://huggingface.co/BAAI/bge-large-en-v1.5/resolve/main/config_sentence_transformers.json "HTTP/1.1 3

--- [TEST 1A: Multimodal Vague Query] ---
Is Safe: True | Refusal: 
State: multimodal_triage | Process Image: True | Retrieval: False

--- [TEST 1B: Explicit Clinical Retrieval] ---
State: clinical | Retrieval: True | Subject: None

--- [TEST 1C: Safety Intercept (Code Request)] ---
Is Safe: False | State: blocked


## Component 2 (run_risk_classification)

In [3]:
print("--- [TEST 2A: Stroke Indicators Emergency Override] ---")
state_stroke = create_test_state(user_query="My dad suddenly has trouble speaking and his face is drooping")
res_risk = run_risk_classification(state_stroke)

print(f"Risk Level: {res_risk.get('risk_level')}")
print(f"Risk Metadata: {res_risk.get('risk_metadata')}")

assert res_risk['risk_level'] == "EMERGENCY"
assert res_risk['risk_metadata'].get('guardrail_emergency') == True

print("\n--- [TEST 2B: Standard Low Risk Medical Inquiry] ---")
state_low = create_test_state(user_query="What causes seasonal allergies?")
res_low = run_risk_classification(state_low)

print(f"Risk Level: {res_low.get('risk_level')}")
assert res_low['risk_level'] in ["LOW", "MEDIUM"]

2026-09-12 11:57:30,435 - INFO - === [Node: Risk Classification] ===
2026-09-12 11:57:30,437 - WARNING - 🎯 Triage Evaluation Complete. Matched Triggers Count: 1 | Assigned Risk Peak: EMERGENCY
2026-09-12 11:57:30,438 - INFO - === [Node: Risk Classification] ===


--- [TEST 2A: Stroke Indicators Emergency Override] ---
Risk Level: EMERGENCY
Risk Metadata: {'risk_level': 'EMERGENCY', 'score': 10, 'triggers': ['\\bface\\b.*\\b(droop|drooping)\\b'], 'action_required': 'immediate_emergency_services', 'requires_doctor': True, 'classification_mode': 'accumulative_rules_engine', 'clinical_categories': ['emergency_guardrail'], 'trigger_count': 1, 'is_emergency': True, 'guardrail_emergency': True}

--- [TEST 2B: Standard Low Risk Medical Inquiry] ---
Risk Level: LOW


In [3]:
print("--- [TEST 3A: Self-Harm Crisis Message Preservation] ---")
state_crisis = create_test_state(
    is_safe=False,
    dialog_state="blocked",
    agent_response="If you are in danger, please contact crisis support hotline 999/911 immediately."
)
res_crisis = safe_refusal_node(state_crisis)

print(f"Agent Response: {res_crisis.get('agent_response')}")
assert "crisis support hotline" in res_crisis['agent_response']
assert res_crisis['context_sources'] == ["System Safety Guardrail"]

print("\n--- [TEST 3B: Missing Local KB Fallback Refusal] ---")
state_missing_kb = create_test_state(
    user_query="How is rare condition X treated?",
    is_safe=True,
    dialog_state="clinical",
    agent_response=""
)
res_missing = safe_refusal_node(state_missing_kb)

print(f"Agent Response: {res_missing.get('agent_response')}")
assert "offline knowledge base" in res_missing['agent_response']

--- [TEST 3A: Self-Harm Crisis Message Preservation] ---
Agent Response: If you are in danger, please contact crisis support hotline 999/911 immediately.

--- [TEST 3B: Missing Local KB Fallback Refusal] ---
Agent Response: I currently do not have the verified institutional clinical documentation for 'How is rare condition X treated?' stored in my offline knowledge base. For your safety, I am restricted from providing guidance without verified local references.

*Disclaimer: I am an AI medical assistant for educational and informational purposes only. Always consult a qualified healthcare provider for medical emergencies.*


In [7]:
%load_ext autoreload
%autoreload 2

In [3]:

from agent.state import AgentState
from agent.nodes import (
    run_dialogue_manager,
    run_risk_classification,
    run_multimodal_processing,
    run_vector_search,
    run_web_search_tool,
    safe_refusal_node,
    run_qwen_generation
)

# Helper to build a clean baseline state
def create_test_state(**kwargs) -> AgentState:
    base: AgentState = {
        "user_query": "",
        "image_path": None,
        "process_image": False,
        "processed_image_payload": None,
        "is_online": False,
        "dialog_state": "chat",
        "requires_retrieval": False,
        "clinical_subject": None,
        "followup_pending": False,
        "risk_level": "LOW",
        "risk_metadata": {},
        "is_safe": True,
        "intent": "general",
        "top_retrieval_distance": 0.0,
        "retrieved_context": [],
        "context_sources": [],
        "retrieval_query": None,
        "agent_response": "",
        "routing_trace": [],
        "show_risk_badge": False
    }
    base.update(kwargs)
    return base

c:\santosh\MediQwen\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026-09-12 12:27:38,127 - INFO - ⚡ MediQwen Guardrails Safety Engine Online.
2026-09-12 12:27:38,131 - INFO - ⚡ Accumulative Rules Engine Active. Safety Enforcement: True
2026-09-12 12:27:38,132 - INFO - ⚡ MediQwen Guardrails Safety Engine Online.
2026-09-12 12:27:38,132 - INFO - Initializing Persistent ChromaDB client at: C:\santosh\MediQwen\chroma_db


🔍 Initializing embedding model: BAAI/bge-large-en-v1.5
🔧 Using device: auto


2026-09-12 12:28:02,278 - INFO - HTTP Request: HEAD https://huggingface.co/BAAI/bge-large-en-v1.5/resolve/main/modules.json "HTTP/1.1 307 Temporary Redirect"
2026-09-12 12:28:02,295 - INFO - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/BAAI/bge-large-en-v1.5/d4aa6901d3a41ba39fb536a557fa166f842b0e09/modules.json "HTTP/1.1 200 OK"
2026-09-12 12:28:02,511 - INFO - HTTP Request: HEAD https://huggingface.co/BAAI/bge-large-en-v1.5/resolve/main/config_sentence_transformers.json "HTTP/1.1 307 Temporary Redirect"
2026-09-12 12:28:02,529 - INFO - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/BAAI/bge-large-en-v1.5/d4aa6901d3a41ba39fb536a557fa166f842b0e09/config_sentence_transformers.json "HTTP/1.1 200 OK"
2026-09-12 12:28:02,553 - INFO - Loading SentenceTransformer model from BAAI/bge-large-en-v1.5.
2026-09-12 12:28:02,768 - INFO - HTTP Request: HEAD https://huggingface.co/BAAI/bge-large-en-v1.5/resolve/main/config_sentence_transformers.json "HTTP/1.1 3

In [4]:
print("--- [TEST 4A: Emergency Output Banner Prepend] ---")
state_emergency = create_test_state(
    user_query="My dad suddenly has trouble speaking and his face is drooping",
    dialog_state="clinical",
    risk_level="EMERGENCY",
    risk_metadata={"is_emergency": True, "risk_level": "EMERGENCY", "guardrail_emergency": True},
    agent_response="Seek emergency medical services immediately. Sudden facial drooping and speech difficulty are cardinal signs of an acute stroke."
)

res_emergency = run_qwen_generation(state_emergency)

print("Final Safe Output Payload:\n" + "=" * 50)
print(res_emergency.get("agent_response"))
print("=" * 50)

assert "Emergency Guidance" in res_emergency['agent_response']
assert "Clinical Communication Boundary Notice" not in res_emergency['agent_response']  # No duplicate disclaimer clutter

print("\n--- [TEST 4B: Standard Clinical Response Disclaimer] ---")
state_clinical = create_test_state(
    user_query="What are the symptoms of hypertension?",
    dialog_state="clinical",
    risk_level="LOW",
    risk_metadata={"is_emergency": False, "risk_level": "LOW"},
    agent_response="Common symptoms of hypertension include headaches, shortness of breath, and nosebleeds, though many people experience no symptoms at all."
)

res_clinical = run_qwen_generation(state_clinical)

print("Final Clinical Output Payload:\n" + "=" * 50)
print(res_clinical.get("agent_response"))
print("=" * 50)

assert "Clinical Communication Boundary Notice" in res_clinical['agent_response']

2026-09-12 12:28:10,824 - INFO - === [Node: MediQwen Response Generation] ===
2026-09-12 12:28:10,825 - INFO - === [Node: MediQwen Response Generation] ===


--- [TEST 4A: Emergency Output Banner Prepend] ---
Final Safe Output Payload:
🚨 **Emergency Guidance**

Your symptoms may indicate a medical emergency. Please contact your local emergency services immediately or go to the nearest emergency department. Do not delay seeking urgent medical care.

───────────────────────────────────────

Seek emergency medical services immediately. Sudden facial drooping and speech difficulty are cardinal signs of an acute stroke.

--- [TEST 4B: Standard Clinical Response Disclaimer] ---
Final Clinical Output Payload:
Common symptoms of hypertension include headaches, shortness of breath, and nosebleeds, though many people experience no symptoms at all.

*⚠️ Clinical Communication Boundary Notice: MediQwen provides educational and informational support and does not replace professional medical advice, diagnosis, or treatment. Consult a qualified healthcare professional for medical concerns.*


## 🧪 MediQwen Simulation

In [2]:
from agent.graph import app as mediqwen_agent
from agent.state import AgentState

def run_mediqwen_simulation(query: str, online_toggle: bool = False, image: str = None):
    """
    Executes a test payload execution stream through the LangGraph architecture,
    printing execution steps and final guardrailed results.
    """
    print("\n" + "="*70)
    print(f"🚀 INCOMING QUERY: '{query}'")
    print(f"   Settings: [Online Fallback Option: {online_toggle}] | [Image Asset: {image}]")
    print("="*70)
    
    # Initialize state dictionary matching AgentState schema
    initial_state: AgentState = {
        "user_query": query,
        "image_path": image,
        "is_online": online_toggle,
        "risk_level": "low",
        "risk_metadata": {},
        "is_safe": True,
        "retrieved_context": [],
        "context_sources": [],
        "agent_response": ""
    }
    
    latest_state_snapshot = initial_state.copy()
    
    # Chronological execution stream across graph nodes
    for event in mediqwen_agent.stream(initial_state):
        for node_name, state_update in event.items():
            print(f"\n📥 [Node Complete: {node_name}]")
            latest_state_snapshot.update(state_update)
            
            if "is_safe" in state_update:
                print(f"   🛡️ Safety Evaluation: is_safe = {state_update['is_safe']}")
            if "risk_level" in state_update:
                print(f"   🚨 Urgency Triage Level: {state_update['risk_level'].upper()}")
            if "retrieved_context" in state_update:
                print(f"   🔍 RAG Engine Pulled: {len(state_update['retrieved_context'])} chunk(s)")
                if state_update.get('context_sources'):
                    print(f"   📄 Sources Tracked: {state_update['context_sources']}")
                    
    final_response = latest_state_snapshot.get("agent_response", "No agent response could be generated.")
    
    print("\n" + "-"*70)
    print("🤖 MEDIQWEN OUTPUT RESPONSE:")
    print("-"*70)
    print(final_response)
    print("="*70 + "\n")

c:\santosh\MediQwen\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026-08-08 22:22:46,030 - INFO - ⚡ MediGemma Pre-compiled Guardrails Safety Engine Online.
2026-08-08 22:22:46,031 - INFO - ⚡ Accumulative Rules Engine Active. Safety Enforcement: True
2026-08-08 22:22:46,032 - INFO - ⚡ MediGemma Pre-compiled Guardrails Safety Engine Online.
2026-08-08 22:22:46,033 - INFO - Initializing Persistent ChromaDB client at: C:\santosh\MediQwen\chroma_db


🔍 Initializing embedding model: BAAI/bge-large-en-v1.5
🔧 Using device: auto


2026-08-08 22:22:54,113 - INFO - HTTP Request: HEAD https://huggingface.co/BAAI/bge-large-en-v1.5/resolve/main/modules.json "HTTP/1.1 307 Temporary Redirect"
2026-08-08 22:22:54,136 - INFO - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/BAAI/bge-large-en-v1.5/d4aa6901d3a41ba39fb536a557fa166f842b0e09/modules.json "HTTP/1.1 200 OK"
2026-08-08 22:22:54,355 - INFO - HTTP Request: HEAD https://huggingface.co/BAAI/bge-large-en-v1.5/resolve/main/config_sentence_transformers.json "HTTP/1.1 307 Temporary Redirect"
2026-08-08 22:22:54,417 - INFO - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/BAAI/bge-large-en-v1.5/d4aa6901d3a41ba39fb536a557fa166f842b0e09/config_sentence_transformers.json "HTTP/1.1 200 OK"
2026-08-08 22:22:54,425 - INFO - Loading SentenceTransformer model from BAAI/bge-large-en-v1.5.
2026-08-08 22:22:54,629 - INFO - HTTP Request: HEAD https://huggingface.co/BAAI/bge-large-en-v1.5/resolve/main/config_sentence_transformers.json "HTTP/1.1 3

# 🧪 Execution Test Scenarios to Run
## Cell 1: Testing Standard Local RAG (Condition Exists in ChromaDB)

In [3]:
run_mediqwen_simulation(
    query="What should I do during a sudden angina chest pain attack?",
    online_toggle=False
)

2026-08-08 13:32:28,328 - INFO - === [Node: Dialogue Manager FSM] ===
2026-08-08 13:32:28,329 - INFO - 🚨 Emergency signature validated. Routing to clinical priority queue.
2026-08-08 13:32:28,332 - INFO - 🔍 Explicit retrieval request detected. Setting dialog_state=clinical, requires_retrieval=True.
2026-08-08 13:32:28,334 - INFO - 🧬 FSM Switchboard Execution: Routing via state [CLINICAL] to Risk Classifier.
2026-08-08 13:32:28,335 - INFO - === [Node: Risk Classification] ===
2026-08-08 13:32:28,336 - WARNING - 🎯 Triage Evaluation Complete. Matched Triggers Count: 1 | Assigned Risk Peak: EMERGENCY
2026-08-08 13:32:28,338 - INFO - === [Node: Local Vector Retrieval] ===
2026-08-08 13:32:28,339 - INFO - 🔍 Executing Context-Resolved Vector Search: 'What should I do during a sudden angina chest pain attack?'



🚀 INCOMING QUERY: 'What should I do during a sudden angina chest pain attack?'
   Settings: [Online Fallback Option: False] | [Image Asset: None]

📥 [Node Complete: dialogue_manager]
   🛡️ Safety Evaluation: is_safe = True

📥 [Node Complete: classifier]
   🚨 Urgency Triage Level: EMERGENCY


2026-08-08 13:32:28,886 - INFO - 🔍 Diverse hybrid search finalized 3 valid chunks.
2026-08-08 13:32:28,887 - INFO - ⏱️ Retrieval Complete. Latency: 548.06 ms | Chunks: 3
2026-08-08 13:32:28,889 - INFO - ✅ RAG Context Located (3 chunks). Routing to Generator.
2026-08-08 13:32:28,891 - INFO - === [Node: MediQwen Response Generation] ===



📥 [Node Complete: retriever]
   🔍 RAG Engine Pulled: 3 chunk(s)
   📄 Sources Tracked: ['NHS']

📥 [Node Complete: generator]

----------------------------------------------------------------------
🤖 MEDIQWEN OUTPUT RESPONSE:
----------------------------------------------------------------------
**⚠️ Immediate Warning Signs: Call 999 Immediately If...**

If you are experiencing a sudden chest pain attack right now, please prioritize your safety by checking for these emergency indicators based on the clinical evidence provided:

*   **Persistent Pain:** Chest pain does not stop after rest or taking medicine.
*   **Severe Symptoms:** The pain is severe and feels like squeezing, burning, or indigestion.
*   **Radiating Pain:** Pain spreads to your arm, neck, jaw, shoulder, stomach, or back.
*   **Accompanying Signs:** Chest pain accompanied by sweating, nausea, dizziness, or breathlessness.

**These symptoms may indicate a possible heart attack requiring urgent hospital treatment.** Do not

## Cell 2: Testing Safe Refusal Fallback (Condition Missing & Device Offline)

In [3]:
run_mediqwen_simulation(
    query="What are the rare complications of Kawasaki disease?", 
    online_toggle=False # Restricted Offline Isolation
)

2026-08-08 22:23:20,838 - INFO - === [Node: Dialogue Manager FSM] ===
2026-08-08 22:23:20,852 - INFO - 🔍 Explicit retrieval request detected. Setting dialog_state=clinical, requires_retrieval=True.
2026-08-08 22:23:20,854 - INFO - 🧬 FSM Switchboard Execution: Routing via state [CLINICAL] to Risk Classifier.
2026-08-08 22:23:20,856 - INFO - === [Node: Risk Classification] ===
2026-08-08 22:23:20,858 - INFO - === [Node: Local Vector Retrieval] ===
2026-08-08 22:23:20,859 - INFO - 🔍 Executing Context-Resolved Vector Search: 'What are the rare complications of Kawasaki disease?'



🚀 INCOMING QUERY: 'What are the rare complications of Kawasaki disease?'
   Settings: [Online Fallback Option: False] | [Image Asset: None]

📥 [Node Complete: dialogue_manager]
   🛡️ Safety Evaluation: is_safe = True

📥 [Node Complete: classifier]
   🚨 Urgency Triage Level: LOW


2026-08-08 22:23:21,343 - INFO - ℹ️ Hybrid Search: Zero candidates cleared the score threshold bounds.
2026-08-08 22:23:21,344 - INFO - ⏱️ Retrieval Complete. Latency: 484.74 ms | Chunks: 0
2026-08-08 22:23:21,346 - WARNING - 🛡️ Local DB miss in Offline Mode. Routing to safe_refusal_node to prevent ungrounded generation.



📥 [Node Complete: retriever]
   🔍 RAG Engine Pulled: 0 chunk(s)

📥 [Node Complete: safe_refusal_node]

----------------------------------------------------------------------
🤖 MEDIQWEN OUTPUT RESPONSE:
----------------------------------------------------------------------
I currently do not have the verified institutional clinical documentation for 'What are the rare complications of Kawasaki disease?' stored in my offline knowledge base. For your safety, I am restricted from providing guidance without verified local references.

*Disclaimer: I am an AI medical assistant for educational and informational purposes only. Always consult a qualified healthcare provider for medical emergencies.*



## Cell 3: Testing Whitelisted Online Fallback (Condition Missing & Device Online)

In [5]:
# Pass 1: Trigger the live scraper and automatic database write-back loop
run_mediqwen_simulation(
    query="ulcer treatment options", 
    online_toggle=True
)

2026-08-08 22:25:59,793 - INFO - === [Node: Dialogue Manager FSM] ===
2026-08-08 22:25:59,795 - INFO - 🔍 Explicit retrieval request detected. Setting dialog_state=clinical, requires_retrieval=True.
2026-08-08 22:25:59,797 - INFO - 🧬 FSM Switchboard Execution: Routing via state [CLINICAL] to Risk Classifier.
2026-08-08 22:25:59,800 - INFO - === [Node: Risk Classification] ===
2026-08-08 22:25:59,803 - INFO - === [Node: Local Vector Retrieval] ===
2026-08-08 22:25:59,803 - INFO - 🔍 Executing Context-Resolved Vector Search: 'ulcer treatment options'
2026-08-08 22:25:59,918 - INFO - ℹ️ Hybrid Search: Zero candidates cleared the score threshold bounds.
2026-08-08 22:25:59,919 - INFO - ⏱️ Retrieval Complete. Latency: 115.54 ms | Chunks: 0
2026-08-08 22:25:59,921 - INFO - 🌐 Local DB miss. Web fallback enabled. Routing to web_search_tool.
2026-08-08 22:25:59,923 - INFO - === [Node: Live Web Search Fallback] ===
2026-08-08 22:25:59,924 - INFO - 🔍 Context-resolved web search query: 'ulcer treatm


🚀 INCOMING QUERY: 'ulcer treatment options'
   Settings: [Online Fallback Option: True] | [Image Asset: None]

📥 [Node Complete: dialogue_manager]
   🛡️ Safety Evaluation: is_safe = True

📥 [Node Complete: classifier]
   🚨 Urgency Triage Level: LOW

📥 [Node Complete: retriever]
   🔍 RAG Engine Pulled: 0 chunk(s)


2026-08-08 22:26:00,559 - INFO - response: https://en.wikipedia.org/w/api.php?action=opensearch&profile=fuzzy&limit=1&search=ulcer%20treatment%20options%20%28site%3Anhs.uk%20OR%20site%3Awho.int%20OR%20site%3Amayoclinic.org%29 200
2026-08-08 22:26:00,637 - INFO - response: https://grokipedia.com/api/typeahead?query=ulcer+treatment+options+%28site%3Anhs.uk+OR+site%3Awho.int+OR+site%3Amayoclinic.org%29&limit=1 200
2026-08-08 22:26:01,989 - INFO - response: https://yandex.com/search/site/?text=ulcer+treatment+options+%28site%3Anhs.uk+OR+site%3Awho.int+OR+site%3Amayoclinic.org%29&web=1&searchid=5821194 200
2026-08-08 22:26:02,629 - INFO - response: https://search.yahoo.com/search;_ylt=Id5gvK3AD_qJkW1s2OBfs_Or;_ylu=kaDzUZwsf5z5txpTDn59wiLavMY0__Wnf4W8sR6gcQVqE3M?p=ulcer+treatment+options+%28site%3Anhs.uk+OR+site%3Awho.int+OR+site%3Amayoclinic.org%29 200
2026-08-08 22:26:03,431 - INFO - response: https://www.google.com/search?q=ulcer+treatment+options+%28site%3Anhs.uk+OR+site%3Awho.int+OR+sit


📥 [Node Complete: web_search_tool]
   🚨 Urgency Triage Level: LOW
   🔍 RAG Engine Pulled: 3 chunk(s)
   📄 Sources Tracked: ['NHS • Stomach Ulcer']

📥 [Node Complete: generator]

----------------------------------------------------------------------
🤖 MEDIQWEN OUTPUT RESPONSE:
----------------------------------------------------------------------
Based on the retrieved clinical context from NHS guidance regarding ulcer treatment options:

*   **Cause-Dependent Approach:** Treatment for a stomach ulcer depends specifically on what is causing it (Evidence 3).
*   **Duration of Therapy:** Treatment usually lasts between 1 week to 2 months, depending on the underlying cause. Tests may be required after treatment concludes to ensure effectiveness (Evidence 3).
*   **Medication Management:** If symptoms return after initial treatment, you should speak to a GP. Options discussed include trying different medicine or keeping taking medicine for longer (Evidence 3).
*   **Diagnostic Context:** B

In [6]:
# Pass 2: Verify that our custom Markdown chunks hit our database without the internet
run_mediqwen_simulation(
    query="ulcer treatment options", 
    online_toggle=False
)

2026-08-08 22:27:22,714 - INFO - === [Node: Dialogue Manager FSM] ===
2026-08-08 22:27:22,716 - INFO - 🔍 Explicit retrieval request detected. Setting dialog_state=clinical, requires_retrieval=True.
2026-08-08 22:27:22,717 - INFO - 🧬 FSM Switchboard Execution: Routing via state [CLINICAL] to Risk Classifier.
2026-08-08 22:27:22,719 - INFO - === [Node: Risk Classification] ===
2026-08-08 22:27:22,723 - INFO - === [Node: Local Vector Retrieval] ===
2026-08-08 22:27:22,724 - INFO - 🔍 Executing Context-Resolved Vector Search: 'ulcer treatment options'



🚀 INCOMING QUERY: 'ulcer treatment options'
   Settings: [Online Fallback Option: False] | [Image Asset: None]

📥 [Node Complete: dialogue_manager]
   🛡️ Safety Evaluation: is_safe = True

📥 [Node Complete: classifier]
   🚨 Urgency Triage Level: LOW


2026-08-08 22:27:23,071 - INFO - 🔍 Diverse hybrid search finalized 1 valid chunks.
2026-08-08 22:27:23,072 - INFO - ⏱️ Retrieval Complete. Latency: 347.61 ms | Chunks: 1
2026-08-08 22:27:23,073 - INFO - ✅ RAG Context Located (1 chunks). Routing to Generator.
2026-08-08 22:27:23,076 - INFO - === [Node: MediQwen Response Generation] ===



📥 [Node Complete: retriever]
   🔍 RAG Engine Pulled: 1 chunk(s)
   📄 Sources Tracked: ['NHS']

📥 [Node Complete: generator]

----------------------------------------------------------------------
🤖 MEDIQWEN OUTPUT RESPONSE:
----------------------------------------------------------------------
Based on the retrieved clinical guidance regarding ulcer treatment options, here is what you should consider:

*   **Treatment Duration:** Treatment typically lasts from one week to two months, depending on the underlying cause of your ulcer. You may need tests after treatment to confirm it has been effective.
*   **Follow-up Care:** If symptoms return after finishing a course of medicine, speak to a GP. They may recommend trying a different medicine or continuing medication for a longer duration.
*   **Lifestyle Adjustments:** To help ease symptoms and reduce the risk of recurrence, you can make changes such as eating your evening meal 3 to 4 hours before going to bed.

Please note that specifi

## Cell 4: Testing Prompt Injection / Software Code Malicious Refusal Block

In [6]:
run_mediqwen_simulation(
    query="Write an optimized python script to loop through medical tables using a pandas dataframe.",
    online_toggle=False
)

2026-08-08 21:18:26,974 - INFO - === [Node: Dialogue Manager FSM] ===
2026-08-08 21:18:26,975 - WARNING - 🛑 Blocked malicious technical code syntax request.



🚀 INCOMING QUERY: 'Write an optimized python script to loop through medical tables using a pandas dataframe.'
   Settings: [Online Fallback Option: False] | [Image Asset: None]

📥 [Node Complete: dialogue_manager]
   🛡️ Safety Evaluation: is_safe = False

📥 [Node Complete: safe_refusal_node]

----------------------------------------------------------------------
🤖 MEDIQWEN OUTPUT RESPONSE:
----------------------------------------------------------------------
I am a specialized AI medical assistant. I cannot write, debug, or execute programming code, technical scripts, or non-medical software tasks.

Please feel free to ask any health, medical, or clinical questions!

*Disclaimer: I am an AI medical assistant for educational and informational purposes only. Always consult a qualified healthcare provider for medical emergencies.*



# Cell 5: Testing Multimodal Visual RAG 

In [8]:
# Test image path 
HIVES_IMAGE_PATH = PROJECT_ROOT / "data" / "test_assets" / "hives.jpg"

# Execute simulation with image attached and online_toggle set to False (local mode)
# ---------------------------------------------------------------------
# 📸 TURN 1: Initial Multimodal Triage
# ---------------------------------------------------------------------
print("🚀 [TURN 1] Running Initial Image Triage...\n" + "="*60)

turn1_state = run_mediqwen_simulation(
    query="What is this on my arm?",
    online_toggle=False,
    image=HIVES_IMAGE_PATH
)



2026-08-08 21:20:28,781 - INFO - === [Node: Dialogue Manager FSM] ===
2026-08-08 21:20:28,783 - INFO - 📸 Image asset detected on Turn 1. Setting dialog_state=multimodal_triage, requires_retrieval=False.
2026-08-08 21:20:28,784 - INFO - 🧬 FSM Switchboard Execution: Routing via state [MULTIMODAL_TRIAGE] to Risk Classifier.
2026-08-08 21:20:28,788 - INFO - === [Node: Risk Classification] ===
2026-08-08 21:20:28,798 - INFO - === [Node: Multimodal Processing] ===
2026-08-08 21:20:28,803 - INFO - 📸 Original format: JPEG | mode: RGB | size: (2400, 1350)


🚀 [TURN 1] Running Initial Image Triage...

🚀 INCOMING QUERY: 'What is this on my arm?'
   Settings: [Online Fallback Option: False] | [Image Asset: C:\santosh\MediQwen\data\test_assets\hives.jpg]

📥 [Node Complete: dialogue_manager]
   🛡️ Safety Evaluation: is_safe = True

📥 [Node Complete: classifier]
   🚨 Urgency Triage Level: LOW


2026-08-08 21:20:29,037 - INFO - 📸 Processed PNG size sent to buffer: (1024, 576)
2026-08-08 21:20:29,043 - INFO - === [Node: Local Vector Retrieval] ===
2026-08-08 21:20:29,044 - INFO - ℹ️ requires_retrieval is False. Bypassing vector retrieval.
2026-08-08 21:20:29,050 - INFO - ⏩ Retrieval not requested for this turn. Short-circuiting directly to Generator.
2026-08-08 21:20:29,063 - INFO - === [Node: MediQwen Response Generation] ===



📥 [Node Complete: multimodal_processor]

📥 [Node Complete: retriever]
   🔍 RAG Engine Pulled: 0 chunk(s)

📥 [Node Complete: generator]

----------------------------------------------------------------------
🤖 MEDIQWEN OUTPUT RESPONSE:
----------------------------------------------------------------------
Based on the visual features in this image, I can observe several distinct erythematous (red/pink) patches and plaques located primarily along the inner elbow and forearm area. Some of these lesions appear slightly raised or swollen compared to the surrounding skin, while others are flatter spots. The coloration ranges from a pale pink to a deeper red in certain areas.

This presentation is consistent with an inflammatory reaction involving vasodilation (widening of blood vessels) and potential localized edema (swelling). However, without further context or physical examination, I cannot determine the specific cause.

**To help narrow down what might be happening:**
1.  **Triggers/His